In [1]:
# ============================================================
# RETAIL PROMPT ENGINEERING DEMO — GOOGLE COLAB
# No API key. No external packages.
# ============================================================

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# ---------- Example demonstrations for one-shot / few-shot ----------
examples = [
    ("My order arrived 3 hours late.", "MEDIUM"),
    ("My order has missing frozen items.", "HIGH"),
    ("The product packaging is slightly damaged but usable.", "LOW"),
]

def classify_issue(text):
    """Simple rule-based simulation of an AI classification."""
    t = text.lower()

    high = [
        "missing", "not received", "wrong item", "fraud",
        "charged twice", "payment", "frozen", "urgent",
        "safety", "unsafe", "cancel"
    ]

    medium = [
        "late", "delay", "damaged", "broken", "defective",
        "poor quality", "incorrect"
    ]

    if any(word in t for word in high):
        return "HIGH"
    elif any(word in t for word in medium):
        return "MEDIUM"
    return "LOW"


def build_prompt(issue, role, context, negative, technique):
    parts = []

    # 1. SYSTEM PROMPT / ROLE
    if role:
        parts.append(
            "SYSTEM PROMPT\n"
            "You are a retail operations and customer-experience analyst. "
            "Classify customer issues by operational priority."
        )

    # 2. ASSISTANT CONTEXT
    if context:
        parts.append(
            "ASSISTANT CONTEXT\n"
            "The customer is reporting an issue with a retail order. "
            "Use only the information provided."
        )

    # 3. ZERO / ONE / FEW SHOT
    if technique == "Zero-shot":
        parts.append(
            "TASK\n"
            "Classify the issue as LOW, MEDIUM, or HIGH priority."
        )

    elif technique == "One-shot":
        parts.append(
            "ONE-SHOT EXAMPLE\n"
            "Example: 'My order has the wrong item.' → HIGH\n\n"
            "TASK\n"
            "Classify the new issue as LOW, MEDIUM, or HIGH priority."
        )

    elif technique == "Few-shot":
        example_text = "\n".join(
            f"Example: '{x}' → {y}" for x, y in examples
        )
        parts.append(
            "FEW-SHOT EXAMPLES\n"
            + example_text +
            "\n\nTASK\n"
            "Classify the new issue as LOW, MEDIUM, or HIGH priority."
        )

    elif technique == "Self-Consistency":
        parts.append(
            "SELF-CONSISTENCY\n"
            "Generate multiple candidate reasoning paths for the same issue "
            "and select the majority final classification."
        )

    # 4. NEGATIVE PROMPTING
    if negative:
        parts.append(
            "NEGATIVE CONSTRAINTS\n"
            "Do not invent order details. "
            "Do not assume a refund is required. "
            "Do not invent company policies."
        )

    # 5. USER MESSAGE
    parts.append("USER MESSAGE\n" + issue)

    return "\n\n".join(parts)


def run_demo(_=None):
    issue = issue_box.value.strip()

    if not issue:
        output.clear_output()
        with output:
            print("Please enter a customer issue.")
        return

    technique = technique_dropdown.value
    role = role_checkbox.value
    context = context_checkbox.value
    negative = negative_checkbox.value

    prompt = build_prompt(
        issue, role, context, negative, technique
    )

    base = classify_issue(issue)

    with output:
        clear_output()

        display(Markdown("## Constructed Prompt"))
        display(Markdown(
            "<pre style='white-space:pre-wrap; "
            "background:#f6f6f6; padding:15px; border-radius:8px;'>"
            + prompt.replace("&", "&amp;").replace("<", "&lt;")
            + "</pre>"
        ))

        display(Markdown("## Model-Style Output"))

        if technique == "Self-Consistency":
            # Simulated multiple sampled paths.
            # Deliberately shows how majority voting works.
            paths = [base, base, base, "MEDIUM", base]

            display(Markdown(
                "\n".join(
                    f"**Path {i+1}:** {answer}"
                    for i, answer in enumerate(paths)
                )
            ))

            counts = {x: paths.count(x) for x in set(paths)}
            final = max(counts, key=counts.get)

            display(Markdown(
                f"### Majority Vote → **{final}**\n\n"
                "Multiple candidate outputs are generated and the most "
                "frequent final answer is selected."
            ))

        else:
            action = {
                "HIGH": "Immediate escalation / priority resolution",
                "MEDIUM": "Standard customer-service resolution workflow",
                "LOW": "Routine handling"
            }[base]

            display(Markdown(
                f"### Priority → **{base}**\n\n"
                f"**Recommended action:** {action}"
            ))

        display(Markdown(
            "---\n"
            "**Prompt engineering demonstrated:** "
            f"{technique} + "
            f"{'Role ' if role else ''}"
            f"{'Context ' if context else ''}"
            f"{'Negative constraints' if negative else ''}"
        ))


# ---------- UI ----------

title = widgets.HTML(
    "<h1>Retail Customer Issue Triage</h1>"
    "<p><b>Prompt Engineering Demonstration</b> — "
    "Explore how different prompting techniques change an AI-style "
    "retail decision workflow.</p>"
)

issue_box = widgets.Textarea(
    value=(
        "My grocery order is two hours late and several "
        "frozen items are missing. I need them for tonight."
    ),
    description="Customer issue:",
    layout=widgets.Layout(width="100%", height="100px"),
    style={"description_width": "initial"}
)

technique_dropdown = widgets.Dropdown(
    options=[
        "Zero-shot",
        "One-shot",
        "Few-shot",
        "Self-Consistency"
    ],
    value="Few-shot",
    description="Technique:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

role_checkbox = widgets.Checkbox(
    value=True,
    description="Role / Persona"
)

context_checkbox = widgets.Checkbox(
    value=True,
    description="Assistant Context"
)

negative_checkbox = widgets.Checkbox(
    value=True,
    description="Negative Prompting"
)

run_button = widgets.Button(
    description="Run Prompt Engineering Demo",
    button_style="primary",
    icon="play"
)

output = widgets.Output()

run_button.on_click(run_demo)

display(title)
display(issue_box)
display(widgets.HBox([
    technique_dropdown,
    role_checkbox,
    context_checkbox,
    negative_checkbox
]))
display(run_button)
display(output)

display(Markdown(
    "> **Teaching note:** This is a local simulation. "
    "It demonstrates prompt construction and majority voting without "
    "calling an LLM API. For a live LLM demonstration, an API model "
    "can later be connected to the same interface."
))


HTML(value='<h1>Retail Customer Issue Triage</h1><p><b>Prompt Engineering Demonstration</b> — Explore how diff…

Textarea(value='My grocery order is two hours late and several frozen items are missing. I need them for tonig…

Button(button_style='primary', description='Run Prompt Engineering Demo', icon='play', style=ButtonStyle())

Output()

> **Teaching note:** This is a local simulation. It demonstrates prompt construction and majority voting without calling an LLM API. For a live LLM demonstration, an API model can later be connected to the same interface.